# Etherpad Pad 内容重建查看器

本工具用于：
1. 查询数据库获取所有 room-id 和版本范围
2. SSH 连接到服务器执行重建脚本
3. 获取并展示重建的内容
4. 支持编辑和对比

---

## 📋 功能清单

- ✅ 查询 MySQL 数据库获取所有 Pad 列表
- ✅ 显示每个 Pad 的版本范围
- ✅ SSH 连接到远程服务器执行重建脚本
- ✅ 查看pad_changes详细内容


## 使用说明

### 完整使用流程

1. **安装依赖** 
2. **导入配置** 
3. **定义函数** 
4. **定义http stream函数** 
5. **初步查询数据库的ROOM基本请开给你** 
6. **分两种方法跑重建的数据**：数据来源于store表，跑线上http stream api重建数据<br>
   6.1.限制pad_id和起始版本号跑重建数据<br>
   6.2.限制起始时间范围跑重建数据
7. **查询的是pad_changes表**：pad_changes记录的每个文档发生增加删除行为的数据

## 1. 安装依赖包

首次使用需要安装必要的 Python 包


In [ ]:
# 安装必要的 Python 包
#!pip install pymysql paramiko pandas ipywidgets -q


## 2. 导入库和配置


In [48]:
import pymysql
import json
import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import re
from datetime import datetime
from collections import defaultdict
import urllib.request
import urllib.parse
import json

# 数据库配置
DB_CONFIG = {
    'host': '112.74.92.135',
    'user': 'root',
    'password': '1q2w3e4R',
    'database': 'alic',
    'charset': 'utf8mb4',
    'port': 3306
}

# 服务器配置（用于 HTTP 请求）
SSH_CONFIG = {
    'host': '8.138.89.124',  # 服务器地址
    'port': 22,
    'username': 'root',
    'password': 'Alic2025'
}

# HTTP API 配置（使用主服务器端口，已集成到主服务器）
HTTP_API_CONFIG = {
    'host': '8.138.89.124',  # 服务器地址
    'port': 9001  # 主服务器端口（默认 9001，与 Etherpad 主服务共享端口）
}

#HTTP_API_CONFIG = {
#    'host': 'localhost',  # 服务器地址
#    'port': 9002  # 主服务器端口（默认 9001，与 Etherpad 主服务共享端口）
#}

print("✅ 配置加载完成（HTTP Stream API 已集成到主服务器，共享端口）")


✅ 配置加载完成（HTTP Stream API 已集成到主服务器，共享端口）


## 3. 定义数据库查询函数


In [49]:
def get_all_room_ids_with_versions():
    """从 store 表查询所有 room-id 及其版本范围"""
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        query = """
        SELECT 
            SUBSTRING_INDEX(SUBSTRING_INDEX(`key`, ':', 2), ':', -1) as pad_id,
            MIN(CAST(SUBSTRING_INDEX(`key`, ':', -1) AS UNSIGNED)) as min_revision,
            MAX(CAST(SUBSTRING_INDEX(`key`, ':', -1) AS UNSIGNED)) as max_revision,
            COUNT(*) as revision_count
        FROM store
        WHERE `key` LIKE 'pad:%:revs:%'
        GROUP BY pad_id
        ORDER BY pad_id
        """
        
        cursor.execute(query)
        results = cursor.fetchall()
        
        cursor.close()
        connection.close()
        
        return results
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}")
        return []

print("✅ 数据库查询函数定义完成")


def query_pads_by_timerange(start_time, end_time):
    """
    根据时间范围查询 Store 数据库中的 Pad 版本信息
    
    参数:
        start_time: 开始时间字符串，格式 'YYYY-MM-DD HH:MM:SS'
        end_time: 结束时间字符串，格式 'YYYY-MM-DD HH:MM:SS'
    
    返回:
        dict: {pad_id: [revision1, revision2, ...]}
    """
    # 转换为时间戳（毫秒）
    start_timestamp = int(datetime.strptime(start_time, '%Y-%m-%d %H:%M:%S').timestamp() * 1000)
    end_timestamp = int(datetime.strptime(end_time, '%Y-%m-%d %H:%M:%S').timestamp() * 1000)
    
    print(f"🔍 查询时间范围: {start_time} ~ {end_time}")
    print(f"📊 时间戳范围: {start_timestamp} ~ {end_timestamp}\n")
    
    # 查询 Store 数据库
    print("🔍 正在查询 Store 数据库...")
    pad_results = []
    
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)

        query = """
        SELECT `key`, `value`
        FROM `store`
        WHERE `key` LIKE 'pad:%%:revs:%%'
          AND JSON_EXTRACT(`value`, '$.meta.timestamp') >= %s
          AND JSON_EXTRACT(`value`, '$.meta.timestamp') <= %s
        ORDER BY `key`
        """
        
        cursor.execute(query, (start_timestamp, end_timestamp))
        pad_results = cursor.fetchall()
        
        print(f"✅ 找到 {len(pad_results)} 条记录\n")
        
        cursor.close()
        connection.close()
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}\n")
        return {}
    
    # 解析 pad_id 和版本号
    print("📊 正在解析 pad_id 和版本号...")
    
    # 正则表达式：pad:room-id:revs:revision
    pattern = re.compile(r'^pad:([^:]+):revs:(\d+)$')
    pad_versions = defaultdict(list)
    
    for row in pad_results:
        key = row['key']
        match = pattern.match(key)
        if match:
            pad_id = match.group(1)
            revision = int(match.group(2))
            pad_versions[pad_id].append(revision)
    
    print(f"✅ 解析完成，共 {len(pad_versions)} 个不同的 Pad")
    
    # 显示每个 Pad 的版本范围
    print("📋 Pad 版本范围统计:")
    for pad_id, revisions in sorted(pad_versions.items()):
        revisions.sort()
        print(f"  - {pad_id}: 版本 {min(revisions)} ~ {max(revisions)} (共 {len(revisions)} 个版本)")
    
    return dict(pad_versions)

✅ 数据库查询函数定义完成


## 4. 定义 SSH 远程执行函数


In [44]:
import base64
import urllib.request
import urllib.parse

def decode_base64_field(value):
    """解码 Base64 字段"""
    if not value:
        return ''
    try:
        return base64.b64decode(value).decode('utf-8')
    except Exception:
        return value  # 如果解码失败，返回原值

def execute_rebuild_script(pad_id, start_rev=None, end_rev=None, use_base64=False, batch_size=None):
    """
    通过 HTTP Stream 获取重建数据（支持自动分批处理）
    
    参数:
        pad_id: Pad ID
        start_rev: 起始版本号
        end_rev: 结束版本号
        use_base64: 是否使用 Base64 编码模式（推荐用于大数据量）
        batch_size: 批次大小（如果指定，则自动分批处理）
    """
    
    # 如果指定了 batch_size，则进行分批处理
    if batch_size and start_rev is not None and end_rev is not None:
        total_versions = end_rev - start_rev + 1
        
        # 如果版本数小于等于批次大小，直接处理
        if total_versions <= batch_size:
            return _execute_single_rebuild(pad_id, start_rev, end_rev, use_base64)
        
        # 需要分批处理
        num_batches = (total_versions + batch_size - 1) // batch_size
        
        print(f"📊 总共 {total_versions} 个版本，将分 {num_batches} 个批次处理（每批 {batch_size} 个）\n")
        
        all_versions = []
        total_success = 0
        total_failed = 0
        
        for batch_idx in range(num_batches):
            batch_start = start_rev + batch_idx * batch_size
            batch_end = min(batch_start + batch_size - 1, end_rev)
            
            print(f"🔄 批次 {batch_idx + 1}/{num_batches}: 版本 {batch_start} - {batch_end}")
            
            # 执行单个批次
            batch_result = _execute_single_rebuild(pad_id, batch_start, batch_end, use_base64)
            
            if batch_result.get('success'):
                batch_versions = batch_result.get('versions', [])
                all_versions.extend(batch_versions)
                
                batch_stats = batch_result.get('statistics', {})
                batch_success = batch_stats.get('success', len(batch_versions))
                batch_failed = batch_stats.get('failed', 0)
                
                total_success += batch_success
                total_failed += batch_failed
                
                print(f"   ✅ 成功: {batch_success}, 失败: {batch_failed}\n")
            else:
                error_msg = batch_result.get('error', 'Unknown error')
                print(f"   ❌ 批次失败: {error_msg}")
                total_failed += (batch_end - batch_start + 1)
        
        # 返回合并后的结果
        return {
            'success': True,
            'pad_id': pad_id,
            'head_revision': end_rev,
            'requested_range': {
                'start': start_rev,
                'end': end_rev
            },
            'statistics': {
                'total': total_versions,
                'success': total_success,
                'failed': total_failed
            },
            'versions': all_versions,
            'batch_info': {
                'batch_size': batch_size,
                'num_batches': num_batches
            }
        }
    else:
        # 不分批，直接处理
        return _execute_single_rebuild(pad_id, start_rev, end_rev, use_base64)


def _execute_single_rebuild(pad_id, start_rev=None, end_rev=None, use_base64=False):
    """
    执行单次重建请求（内部函数）
    
    参数:
        pad_id: Pad ID
        start_rev: 起始版本号
        end_rev: 结束版本号
        use_base64: 是否使用 Base64 编码模式
    """
    try:
        # 构建 HTTP 请求 URL
        params = {
            'padId': pad_id,
            'useBase64': 'true' if use_base64 else 'false'
        }
        
        if start_rev is not None:
            params['startRev'] = str(start_rev)
        if end_rev is not None:
            params['endRev'] = str(end_rev)
        
        url = f"http://{HTTP_API_CONFIG['host']}:{HTTP_API_CONFIG['port']}/api/rebuild?{urllib.parse.urlencode(params)}"
        
        # 发送 HTTP 请求并流式读取响应
        req = urllib.request.Request(url)
        
        with urllib.request.urlopen(req, timeout=600) as response:
            # 解析 NDJSON 流式响应
            all_versions = []
            header_info = None
            statistics = None
            
            # 逐行读取 NDJSON 格式的响应
            for line_bytes in response:
                line = line_bytes.decode('utf-8').strip()
                if not line:
                    continue
                
                try:
                    data = json.loads(line)
                    
                    # 第一行是头部信息
                    if 'stream' in data and data.get('stream'):
                        header_info = data
                        continue
                    
                    # 最后一行是统计信息
                    if '_statistics' in data:
                        statistics = data.get('_statistics', {})
                        continue
                    
                    # 版本数据
                    if 'revision' in data:
                        all_versions.append(data)
                        
                except json.JSONDecodeError as e:
                    continue
            
            # 构建结果对象
            result = {
                'success': True,
                'pad_id': pad_id,
                'head_revision': header_info.get('head_revision') if header_info else None,
                'requested_range': header_info.get('requested_range', {}) if header_info else {},
                'statistics': statistics or {},
                'versions': all_versions
            }
            
            # 处理 Base64 解码
            if header_info and header_info.get('encoding') == 'base64':
                result['encoding'] = 'base64'
                for version in all_versions:
                    if 'content_base64' in version:
                        version['content'] = decode_base64_field(version['content_base64'])
                        del version['content_base64']
                    if 'changeset_base64' in version:
                        version['changeset'] = decode_base64_field(version['changeset_base64'])
                        del version['changeset_base64']
                    if 'attribs_base64' in version:
                        version['attribs'] = decode_base64_field(version['attribs_base64'])
                        del version['attribs_base64']
            
            return result
            
    except urllib.error.HTTPError as e:
        error_body = e.read().decode('utf-8') if e.fp else ''
        try:
            error_data = json.loads(error_body)
            return {'success': False, 'error': error_data.get('error', str(e))}
        except:
            return {'success': False, 'error': f'HTTP {e.code}: {e.reason}'}
            
    except urllib.error.URLError as e:
        return {'success': False, 'error': f'Connection error: {e.reason}'}
        
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {'success': False, 'error': str(e)}


print("✅ HTTP Stream 执行函数定义完成（支持自动分批处理）")

✅ HTTP Stream 执行函数定义完成（支持自动分批处理）


## 5. 查询所有 Room ID 和版本范围

初步查一查文档的基本情况

In [4]:
# 查询所有 room-id 及版本范围
print("🔍 正在查询数据库...")
room_list = get_all_room_ids_with_versions()

if room_list:
    df = pd.DataFrame(room_list)
    df.index = df.index + 1
    
    print(f"\n✅ 找到 {len(room_list)} 个 Pad\n")
    display(df)
    
    # 统计信息
    total_revisions = df['revision_count'].sum()
    avg_revisions = df['revision_count'].mean()
    
    print(f"\n📊 统计信息:")
    print(f"   总 Pad 数: {len(room_list)}")
    print(f"   总版本数: {total_revisions}")
    print(f"   平均版本数: {avg_revisions:.2f}")
    print(f"   最多版本: {df['revision_count'].max()} (Pad: {df.loc[df['revision_count'].idxmax(), 'pad_id']})")
    print(f"   最少版本: {df['revision_count'].min()} (Pad: {df.loc[df['revision_count'].idxmin(), 'pad_id']})")
else:
    print("❌ 未找到任何 Pad 数据")
    df = pd.DataFrame()


🔍 正在查询数据库...

✅ 找到 13 个 Pad



,pad_id,min_revision,max_revision,revision_count
1,room-165,0,0,1
2,room-207,0,0,1
3,room-208,0,24,25
4,room-210,0,0,1
5,room-224,0,0,1
6,room-229,0,200,201
7,room-243,0,29,30
8,room-254,0,0,1
9,room-255,0,0,1
10,room-258,0,8,9



📊 统计信息:
   总 Pad 数: 13
   总版本数: 1244
   平均版本数: 95.69
   最多版本: 735 (Pad: room-262)
   最少版本: 1 (Pad: room-165)


## 6. 执行 Pad 内容重建

### 6.1 按照pad_id和起始版本号复原共享文档数据

💡 **使用说明：** 修改下面代码中的参数，然后运行单元格

- `pad_id`: 从上面表格中选择一个 Pad ID
- `start_rev`: 起始版本号
- `end_rev`: 结束版本号
- `batch_size`:每次请求的版本数量

In [47]:
# 📝 在这里修改参数
pad_id = 'room-262'        # 修改为你要查询的 Pad ID
start_rev = 0              # 起始版本号
end_rev = 734              # 结束版本号
use_base64 = True          # 使用 Base64 编码模式（推荐用于大数据量和远程调用）
batch_size = 20            # 每次请求的版本数量（设为 None 则不分批）

# 执行重建
print(f"{'='*80}")
print(f"📝 Pad ID: {pad_id}")
print(f"📊 版本范围: {start_rev} - {end_rev}")
print(f"🔧 编码模式: {'Base64（推荐）' if use_base64 else '标准模式'}")
print(f"📦 批次大小: {'每次请求 ' + str(batch_size) + ' 个版本' if batch_size else '不分批（一次性处理）'}")
print(f"⏰ 开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

# 调用函数（自动处理分批逻辑）
rebuild_result = execute_rebuild_script(
    pad_id=pad_id,
    start_rev=start_rev,
    end_rev=end_rev,
    use_base64=use_base64,
    batch_size=batch_size  # 传入 batch_size，函数会自动分批
)

# 显示结果
if rebuild_result.get('success'):
    stats = rebuild_result.get('statistics', {})
    batch_info = rebuild_result.get('batch_info', {})
    
    print(f"{'='*80}")
    print(f"✅ 重建完成！")
    print(f"{'='*80}")
    print(f"   总版本数: {stats.get('total', len(rebuild_result.get('versions', [])))}")
    print(f"   成功: {stats.get('success', 0)}")
    print(f"   失败: {stats.get('failed', 0)}")
    
    if batch_info:
        print(f"   批次数: {batch_info.get('num_batches', 1)}")
        print(f"   批次大小: {batch_info.get('batch_size', 'N/A')}")
    
    print(f"⏰ 完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"💡 结果已保存到变量 rebuild_result，可以在下面的单元格中查看详细内容")
else:
    print(f"❌ 重建失败: {rebuild_result.get('error', 'Unknown error')}")

📝 Pad ID: room-262
📊 版本范围: 0 - 734
🔧 编码模式: Base64（推荐）
📦 批次大小: 每次请求 20 个版本
⏰ 开始时间: 2026-01-20 11:36:49

📊 总共 735 个版本，将分 37 个批次处理（每批 20 个）

🔄 批次 1/37: 版本 0 - 19
   ✅ 成功: 20, 失败: 0

🔄 批次 2/37: 版本 20 - 39
   ✅ 成功: 20, 失败: 0

🔄 批次 3/37: 版本 40 - 59
   ✅ 成功: 20, 失败: 0

🔄 批次 4/37: 版本 60 - 79
   ✅ 成功: 20, 失败: 0

🔄 批次 5/37: 版本 80 - 99
   ✅ 成功: 20, 失败: 0

🔄 批次 6/37: 版本 100 - 119
   ✅ 成功: 20, 失败: 0

🔄 批次 7/37: 版本 120 - 139
   ✅ 成功: 20, 失败: 0

🔄 批次 8/37: 版本 140 - 159
   ✅ 成功: 20, 失败: 0

🔄 批次 9/37: 版本 160 - 179
   ✅ 成功: 20, 失败: 0

🔄 批次 10/37: 版本 180 - 199
   ✅ 成功: 20, 失败: 0

🔄 批次 11/37: 版本 200 - 219
   ✅ 成功: 20, 失败: 0

🔄 批次 12/37: 版本 220 - 239
   ✅ 成功: 20, 失败: 0

🔄 批次 13/37: 版本 240 - 259
   ✅ 成功: 20, 失败: 0

🔄 批次 14/37: 版本 260 - 279
   ✅ 成功: 20, 失败: 0

🔄 批次 15/37: 版本 280 - 299
   ✅ 成功: 20, 失败: 0

🔄 批次 16/37: 版本 300 - 319
   ✅ 成功: 20, 失败: 0

🔄 批次 17/37: 版本 320 - 339
   ✅ 成功: 6, 失败: 14

🔄 批次 18/37: 版本 340 - 359
   ✅ 成功: 20, 失败: 0

🔄 批次 19/37: 版本 360 - 379
   ✅ 成功: 20, 失败: 0

🔄 批次 20/37: 版本 380 - 399
  

### 6.2 按照时间范围复原共享文档数据

💡 **使用说明：** 修改下面代码中的参数，然后运行单元格

- `start_time`: 开始时间
- `end_time`: 结束时间
- `batch_size`:每次请求的版本数量

In [52]:
# ==================== 使用示例 ====================
# 配置参数
start_time = '2025-12-24 09:58:39'  # 开始时间 
end_time = '2025-12-24 19:59:47'    # 结束时间
batch_size = 20  # 批量大小

# API 端点
api_url = f"http://{HTTP_API_CONFIG['host']}:{HTTP_API_CONFIG['port']}/api/rebuild"

# 调用函数查询
pad_versions_with_time = query_pads_by_timerange(start_time, end_time)

🔍 查询时间范围: 2025-12-24 09:58:39 ~ 2025-12-24 19:59:47
📊 时间戳范围: 1766541519000 ~ 1766577587000

🔍 正在查询 Store 数据库...
✅ 找到 542 条记录

📊 正在解析 pad_id 和版本号...
✅ 解析完成，共 7 个不同的 Pad
📋 Pad 版本范围统计:
  - room-207: 版本 0 ~ 0 (共 1 个版本)
  - room-208: 版本 0 ~ 24 (共 25 个版本)
  - room-229: 版本 160 ~ 165 (共 6 个版本)
  - room-243: 版本 27 ~ 29 (共 3 个版本)
  - room-260: 版本 47 ~ 208 (共 162 个版本)
  - room-261: 版本 0 ~ 20 (共 21 个版本)
  - room-262: 版本 0 ~ 323 (共 324 个版本)


In [53]:
# ==================== 批量调用重建 API ====================
print(f"\n🚀 开始批量重建 (批量大小: {batch_size})...")

all_results = []
total_processed = 0
total_pads = len(pad_versions_with_time)

for idx, (pad_id, revisions) in enumerate(sorted(pad_versions_with_time.items()), 1):
    revisions.sort()
    start_rev = min(revisions)
    end_rev = max(revisions)
    
    print(f"\n[{idx}/{total_pads}] 处理 Pad: {pad_id}")
    print(f"  版本范围: {start_rev} ~ {end_rev}")
    
    # 调用已有的 execute_rebuild_script 函数
    result = execute_rebuild_script(
        pad_id=pad_id,
        start_rev=start_rev,
        end_rev=end_rev,
        use_base64=False,  # 可以根据需要改为 True
        batch_size=batch_size
    )
    
    # 检查结果
    if result.get('success'):
        versions = result.get('versions', [])
        all_results.extend(versions)
        total_processed += len(versions)
        print(f"  ✅ 成功重建 {len(versions)} 个版本")
    else:
        error_msg = result.get('error', '未知错误')
        print(f"  ❌ 重建失败: {error_msg}")

# ==================== 结果统计 ====================
print("" + "="*60)
print("📈 批量重建完成统计")
print("="*60)
print(f"处理的 Pad 数量: {total_pads}")
print(f"成功重建的版本数: {total_processed}")
print(f"平均每个 Pad 的版本数: {total_processed / total_pads if total_pads > 0 else 0:.2f}")


🚀 开始批量重建 (批量大小: 20)...

[1/7] 处理 Pad: room-207
  版本范围: 0 ~ 0
  ✅ 成功重建 1 个版本

[2/7] 处理 Pad: room-208
  版本范围: 0 ~ 24
📊 总共 25 个版本，将分 2 个批次处理（每批 20 个）

🔄 批次 1/2: 版本 0 - 19
   ✅ 成功: 20, 失败: 0

🔄 批次 2/2: 版本 20 - 24
   ✅ 成功: 5, 失败: 0

  ✅ 成功重建 25 个版本

[3/7] 处理 Pad: room-229
  版本范围: 160 ~ 165
  ✅ 成功重建 6 个版本

[4/7] 处理 Pad: room-243
  版本范围: 27 ~ 29
  ✅ 成功重建 3 个版本

[5/7] 处理 Pad: room-260
  版本范围: 47 ~ 208
📊 总共 162 个版本，将分 9 个批次处理（每批 20 个）

🔄 批次 1/9: 版本 47 - 66
   ✅ 成功: 20, 失败: 0

🔄 批次 2/9: 版本 67 - 86
   ✅ 成功: 20, 失败: 0

🔄 批次 3/9: 版本 87 - 106
   ✅ 成功: 20, 失败: 0

🔄 批次 4/9: 版本 107 - 126
   ✅ 成功: 20, 失败: 0

🔄 批次 5/9: 版本 127 - 146
   ✅ 成功: 20, 失败: 0

🔄 批次 6/9: 版本 147 - 166
   ✅ 成功: 20, 失败: 0

🔄 批次 7/9: 版本 167 - 186
   ✅ 成功: 20, 失败: 0

🔄 批次 8/9: 版本 187 - 206
   ✅ 成功: 20, 失败: 0

🔄 批次 9/9: 版本 207 - 208
   ✅ 成功: 2, 失败: 0

  ✅ 成功重建 162 个版本

[6/7] 处理 Pad: room-261
  版本范围: 0 ~ 20
📊 总共 21 个版本，将分 2 个批次处理（每批 20 个）

🔄 批次 1/2: 版本 0 - 19
   ✅ 成功: 20, 失败: 0

🔄 批次 2/2: 版本 20 - 20
   ✅ 成功: 1, 失败: 0

  ✅ 成功重建 21 个版本

[

In [54]:
timerange_versions = [v for v in timerange_rebuild_results if 'revision' in v]

# 直接转 DataFrame
timerange_rebuild_df = pd.DataFrame(timerange_versions)

# 设置 Pandas 显示选项
#pd.set_option('display.max_rows', None)  # 显示所有行
#pd.set_option('display.max_columns', None)  # 显示所有列
#pd.set_option('display.max_colwidth', 100)  # 列内容最多显示100字符（可调整）
#pd.set_option('display.width', None)  # 自动调整宽度

# 显示结果
print(f"\n✅ 已转换为 DataFrame，共 {len(timerange_rebuild_df)} 条版本记录")
print(f"📊 列名: {list(timerange_rebuild_df.columns)}")


✅ 已转换为 DataFrame，共 323 条版本记录
📊 列名: ['revision', 'success', 'pad_id', 'author', 'timestamp', 'formatted_timestamp', 'text_length', 'line_count', 'change_summary', 'content', 'changeset', 'attribs']


In [37]:
timerange_rebuild_df.head()

,revision,success,pad_id,author,timestamp,formatted_timestamp,text_length,line_count,change_summary,content,changeset,attribs
0,1,True,room-262,a.VtElYwUzETeRjnsZ,1766577245149,2025-12-24 19:54:05.149,229,7,229 -> 230 chars,Welcome to Etherpad!\n\nThis pad text is synch...,Z:6d>1|4=4x=1e*0|1+1$\n,|4+4x+1e*0|1+1|2+2
1,2,True,room-262,a.VtElYwUzETeRjnsZ,1766577245638,2025-12-24 19:54:05.638,230,8,230 -> 231 chars,Welcome to Etherpad!\n\nThis pad text is synch...,Z:6e>1|5=6c*0|1+1$\n,|4+4x+1e*0|2+2|2+2
2,3,True,room-262,a.VtElYwUzETeRjnsZ,1766577246344,2025-12-24 19:54:06.344,232,8,231 -> 233 chars,Welcome to Etherpad!\n\nThis pad text is synch...,Z:6f>2|6=6d*0+2$杨沁,|4+4x+1e*0|2+2*0+2|2+2
3,4,True,room-262,a.VtElYwUzETeRjnsZ,1766577246941,2025-12-24 19:54:06.941,233,8,233 -> 234 chars,Welcome to Etherpad!\n\nThis pad text is synch...,Z:6h>1|6=6d=2*0+1$：,|4+4x+1e*0|2+2*0+3|2+2
4,5,True,room-262,a.VtElYwUzETeRjnsZ,1766577247744,2025-12-24 19:54:07.744,234,8,234 -> 235 chars,Welcome to Etherpad!\n\nThis pad text is synch...,Z:6i>1|6=6d=3*0+1$我,|4+4x+1e*0|2+2*0+4|2+2


## 7. 读取 pad_version_changes 表数据

pad_version_changes通过store数据解析出来，记录变化的数据，add为添加操作，delete为删除操作，从数据库中读取指定 pad 的版本变化记录

In [ ]:
def get_pad_version_changes(room_id=None):
    """从数据库中读取指定 pad 的版本变化记录"""
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        if room_id:
            query = """SELECT * FROM pad_version_changes WHERE pad_id = %s ORDER BY seq_order ASC"""
            cursor.execute(query, [room_id])
        else:
            query = """SELECT * FROM pad_version_changes ORDER BY pad_id, seq_order ASC"""
            cursor.execute(query)
        
        results = cursor.fetchall()
        
        cursor.close()
        connection.close()
        
        return results
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}")
        return []

print("✅ 数据库查询函数定义完成")

✅ 数据库查询函数定义完成


In [ ]:
# 读取数据
room_id = "room-229"
df_versions = get_pad_version_changes(room_id=room_id)
df_versions = pd.DataFrame(df_versions)
if df_versions is not None and len(df_versions) > 0:
    df = df_versions[[
    "pad_id",
    "seq_order",
    "behavior",
    "author",
    "content",
    "add_start_time",
    "add_end_time",
    "delete_start_time",
    "delete_end_time"
]]

df_versions.head(5)

,pad_id,seq_order,behavior,author,content,add_start_time,add_end_time,delete_start_time,delete_end_time
0,room-229,1,deleted,a.rVEwX679hNTTNivd,*,2025-09-26 23:05:32.012,2025-09-26 23:05:32.012,2025-10-21 22:47:16.498,2025-10-21 22:47:16.498
1,room-229,2,add,a.rVEwX679hNTTNivd,欢迎来到,2025-10-21 22:47:34.952,2025-10-21 22:47:34.952,None,None
2,room-229,3,add,a.ni6xvsCFoJs9Rr1v,Welcome to Etherpad!\n\n,2025-09-17 23:39:12.431,2025-09-17 23:39:12.431,None,None
3,room-229,4,deleted,a.ni6xvsCFoJs9Rr1v,"This pad text is synchronized as you type, so ...",2025-09-17 23:39:12.431,2025-09-17 23:39:12.431,2025-09-20 22:15:52.139,2025-09-20 22:15:52.139
4,room-229,5,add,a.ni6xvsCFoJs9Rr1v,周末去哪里玩\n可以去迪士尼,2025-09-20 22:15:58.322,2025-09-20 22:16:11.664,None,None
